# TensorTrade: Backtest model predictions as trading signal

Use **TensorTrade** to simulate trading on our Lag+Ridge (and optionally LSTM) predictions. The environment handles execution, commission, and P&L; we apply a **threshold policy** (trade when predicted return > threshold).

**Goal:** See if our existing predictor improves **trading outcomes** (P&L vs buy-and-hold) when used as a signal inside TensorTrade's execution simulation.

**Using with AllInOne v7:** Use the **same `ASSET`** in both notebooks. Run [Crypto_Colab_AllInOne_v7.ipynb](Crypto_Colab_AllInOne_v7.ipynb) for prediction metrics (MAE, RMSE, directional accuracy, ablation, interpretability). Run this notebook for **trading P&L** (model-threshold vs buy-and-hold vs cash). Together: v7 tells you how well the model predicts; this notebook tells you how that translates to simulated trading.

In [ ]:
# Config: same asset as AllInOne v7
ASSET = "ETC-USD"   # or "ETH-USD", "BTC-USD", "XRP-USD"
COMMISSION = 0.0005  # 0.05% per trade (lower = less drag; try 0.001 or 0)
PRED_RET_THRESHOLD = 0.01   # trade when |predicted return| > this (0.5%→more trades, 1%→fewer; raise to improve)
# Optional: asymmetric thresholds to reduce sell bias (if set, override PRED_RET_THRESHOLD)
BUY_THRESHOLD = None   # e.g. 0.008 = buy when pred_ret > 0.8%
SELL_THRESHOLD = None  # e.g. 0.015 = sell only when pred_ret < -1.5% (fewer sells)

## 1. Install TensorTrade (optional)

Uncomment and run once. **You will see "ERROR: pip's dependency resolver ... dependency conflicts" and a list of packages.** That is normal on Colab—no numpy/pandas version satisfies every preinstalled package. **Ignore those messages.** Do **not** run the "fix" cell below. Go to **Section 2** and run the next cells. If `import pandas` and the backtest run without crashing, you're done.

In [ ]:
# Uncomment and run once. Ignore any "ERROR: ... dependency conflicts" after this.
# !pip install -q --upgrade pip
# !pip install -q gymnasium
# !pip install -q git+https://github.com/tensortrade-org/tensortrade.git

### Colab: "ERROR: ... dependency conflicts" — ignore them

After the install cell you'll see conflicts (e.g. google-colab wants pandas 2.2.2, jax wants numpy≥2, tensortrade/stochastic want other versions). **No single numpy/pandas version satisfies all of them.** Do nothing: go to Section 2 and run the notebook. If it runs, you're fine.

**Only if the notebook crashes** (e.g. `ValueError: numpy.dtype size changed` on `import pandas`): run the cell below, restart runtime, re-run. You'll get different conflict messages; if the notebook still runs, ignore those too.

In [ ]:
# Only if you crashed with "numpy.dtype size changed". Then restart runtime and re-run.
# This may create other conflicts (stochastic, tensorflow); if the notebook still runs, ignore them.
!pip install --upgrade pip -q
!pip install --force-reinstall "numpy>=2" -q
!pip install --force-reinstall "pandas==2.2.2" -q
print("Done. Runtime → Restart session, then re-run from the top.")

In [ ]:
# Option B (local only): NumPy 1.x — use only if no jax/opencv/shap and you want numpy<2
# !pip install --upgrade pip -q
# !pip install --force-reinstall "numpy>=1.26.4,<2.0" -q
# !pip install --force-reinstall "pandas==2.2.2" -q
# print("Done (NumPy 1.x). Restart kernel, then re-run from the top.")

## 2. Data and Lag+Ridge model (same as AllInOne v7)

Load data, build train/val/test split, train Lag+Ridge, and compute **predicted return** for the test period. We use this as the trading signal.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

DATA_DIR = Path("/content/data") if Path("/content").exists() else Path("./data")
DATA_DIR.mkdir(exist_ok=True)

cache_name = ASSET.replace("-", "_") + "_daily.parquet"
cache_path = DATA_DIR / cache_name
if cache_path.exists():
    df = pd.read_parquet(cache_path)
else:
    raw = yf.download(ASSET, start="2017-01-01", end=None, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index().ffill().dropna()
    df = raw[["Close"]].copy()
    df.columns = ["price"]
    if "Volume" in raw.columns:
        df["volume"] = raw["Volume"]
    df.to_parquet(cache_path)
if "volume" not in df.columns:
    df["volume"] = 0.0
df["ret"] = (df["price"] - df["price"].shift(1)) / (df["price"].shift(1) + 1e-12)
df["volatility_14"] = df["ret"].rolling(14).std()
df["log_volume"] = np.log1p(df["volume"])

n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
train_df = df.iloc[:train_end]
test_df = df.iloc[val_end:]
price_full = df["price"].values
log_vol_full = df["log_volume"].values
vol_full = np.nan_to_num(df["volatility_14"].values, nan=0.0)

N_LAGS = 30
def build_lag_features(price, n_lags):
    T = len(price)
    X_list = [price[n_lags - lag : T - lag] for lag in range(1, n_lags + 1)]
    X = np.column_stack(X_list)[:-1]
    y = price[n_lags + 1 :]
    return X, y

def add_vol_volatility(X, y, start_idx, n_lags):
    n = len(y)
    idx = start_idx + n_lags
    extra = np.column_stack([log_vol_full[idx : idx + n], vol_full[idx : idx + n]])
    return np.hstack([X, extra])

X_train, y_train = build_lag_features(train_df["price"].values, N_LAGS)
X_test, y_test = build_lag_features(test_df["price"].values, N_LAGS)
X_train = add_vol_volatility(X_train, y_train, 0, N_LAGS)
X_test = add_vol_volatility(X_test, y_test, val_end, N_LAGS)

n_f = X_train.shape[1]
pipe = Pipeline([
    ("scale", ColumnTransformer([("s", StandardScaler(), list(range(n_f)))], remainder="passthrough")),
    ("ridge", Ridge(alpha=1.0)),
])
pipe.fit(X_train, y_train)
pred_lag = pipe.predict(X_test)
print("Lag+Ridge test predictions:", len(pred_lag))

## 3. Build TensorTrade environment and backtest with threshold policy

We feed **close**, **volume**, **volatility_14**, and **model predicted return** into the env. At each step we choose: BUY if pred_ret > threshold, SELL if pred_ret < -threshold, else HOLD. TensorTrade simulates execution and commission and tracks P&L.

**Why the comparison can look odd:** The model-threshold strategy *starts in 100% cash* and only buys when the signal is strong. Buy-and-hold is *100% invested from day one*. So in a period where the asset crashes (e.g. -70%), the model "beats" buy-and-hold simply because it held cash—not because the predictions were good. We also report **cash** (0% in asset): if the model loses 4% and cash loses 0%, the model did *worse* than doing nothing. So we compare all three: model P&L vs buy-and-hold vs cash.

In [ ]:
# Align backtest window: "current" price when we have prediction j
start_idx = val_end + N_LAGS
close_arr = price_full[start_idx : start_idx + len(y_test)]
vol_arr = vol_full[start_idx : start_idx + len(y_test)]
log_vol_arr = log_vol_full[start_idx : start_idx + len(y_test)]
pred_ret_arr = (pred_lag - close_arr) / (close_arr + 1e-12)

# For TensorTrade we need lists (or arrays) of same length
close_list = close_arr.tolist()
volume_list = (np.expm1(log_vol_arr)).tolist()  # approximate volume for display
vol_list = vol_arr.tolist()
pred_ret_list = pred_ret_arr.tolist()
T = len(close_list)
print("Backtest length:", T, "steps")

In [ ]:
try:
    from tensortrade.feed.core import DataFeed, Stream
    from tensortrade.oms.exchanges import Exchange, ExchangeOptions
    from tensortrade.oms import instruments as tt_instruments
    from tensortrade.oms.services.execution.simulated import execute_order
    from tensortrade.oms.wallets import Wallet, Portfolio
    from tensortrade.env.default.actions import BSH
    from tensortrade.env.default.rewards import PBR
    import tensortrade.env.default as default
    _sym = ASSET.split("-")[0]
    ASSET_INSTRUMENT = getattr(tt_instruments, _sym, tt_instruments.BTC)
    TENSORTRADE_AVAILABLE = True
except ImportError as e:
    TENSORTRADE_AVAILABLE = False
    print("TensorTrade not installed. Run: pip install gymnasium && pip install git+https://github.com/tensortrade-org/tensortrade.git")
    print("Error:", e)

In [ ]:
if TENSORTRADE_AVAILABLE:
    _pair = f"USD-{ASSET.split('-')[0]}"
    price = Stream.source(close_list, dtype="float").rename(_pair)
    exchange_options = ExchangeOptions(commission=COMMISSION)
    exchange = Exchange("exchange", service=execute_order, options=exchange_options)(price)
    initial_cash = 10000.0
    cash = Wallet(exchange, initial_cash * tt_instruments.USD)
    asset = Wallet(exchange, 0 * ASSET_INSTRUMENT)
    portfolio = Portfolio(tt_instruments.USD, [cash, asset])

    features = [
        Stream.source(close_list, dtype="float").rename("close"),
        Stream.source(volume_list, dtype="float").rename("volume"),
        Stream.source(vol_list, dtype="float").rename("volatility_14"),
        Stream.source(pred_ret_list, dtype="float").rename("model_pred_ret"),
    ]
    feed = DataFeed(features)
    feed.compile()

    reward_scheme = PBR(price=price)
    action_scheme = BSH(cash=cash, asset=asset).attach(reward_scheme)
    env = default.create(
        feed=feed,
        portfolio=portfolio,
        action_scheme=action_scheme,
        reward_scheme=reward_scheme,
        window_size=5,
        max_allowed_loss=0.5,
    )
    print("Environment created. Window size 5, commission", COMMISSION)

In [ ]:
if TENSORTRADE_AVAILABLE:
    obs, info = env.reset()
    done = truncated = False
    step = 0
    total_reward = 0.0
    n_buys = n_sells = 0
    # Threshold policy: 0=BUY, 1=SELL, 2=HOLD
    while not (done or truncated) and step < T:
        pred_ret = pred_ret_list[step]
        buy_thresh = BUY_THRESHOLD if BUY_THRESHOLD is not None else PRED_RET_THRESHOLD
        sell_thresh = SELL_THRESHOLD if SELL_THRESHOLD is not None else PRED_RET_THRESHOLD
        if pred_ret > buy_thresh:
            action = 0  # BUY
            n_buys += 1
        elif pred_ret < -sell_thresh:
            action = 1  # SELL
            n_sells += 1
        else:
            action = 2  # HOLD
        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward) if np.isscalar(reward) else float(np.asarray(reward).item())
        step += 1

    final_worth = portfolio.net_worth
    pnl = final_worth - initial_cash
    pnl_pct = 100 * (pnl / initial_cash)
    # Buy-and-hold = 100% in asset from first to last day of backtest
    buy_hold_return = (close_list[-1] - close_list[0]) / (close_list[0] + 1e-12)
    buy_hold_pnl = initial_cash * buy_hold_return
    # Cash baseline = do nothing (0% return)
    cash_pnl = 0.0

    print("--- Model-threshold policy (TensorTrade) ---")
    print(f"  Steps: {step}  Buys: {n_buys}  Sells: {n_sells}  Total reward: {total_reward:.2f}")
    print(f"  Final portfolio value: ${final_worth:,.2f}  P&L: ${pnl:+,.2f} ({pnl_pct:+.2f}%)")
    print("--- Buy-and-hold (100% in asset from start to end) ---")
    print(f"  P&L: ${buy_hold_pnl:+,.2f} ({100*buy_hold_return:+.2f}%)")
    print("--- Cash (0% in asset the whole time) ---")
    print(f"  P&L: ${cash_pnl:+,.2f} (0.00%)")
    print("---")
    # Interpretation: model starts in CASH and only buys when signal > threshold.
    # So in a big downtrend, the model often stays in cash and loses less than buy-and-hold.
    # That does NOT mean the model added value: compare to cash. If model P&L < 0, cash (0%) did better.
    if pnl > buy_hold_pnl and pnl > cash_pnl:
        print("Model-threshold beat both buy-and-hold and cash (model added value).")
    elif pnl > buy_hold_pnl:
        print("Model lost less than buy-and-hold (held more cash in a downtrend). Versus cash, model lost money.")
    elif pnl > cash_pnl:
        print("Model beat cash but lost to buy-and-hold (period was up; model stayed in cash too much).")
    else:
        print("Model lost to both: buy-and-hold and cash did better.")

## 4. Improving results — what to try

**Your run had 88 buys, 304 sells (−4% P&L vs 0% cash).** Main issues: (1) **too many trades** → commission drag; (2) **sell bias** in a downtrend. Try:

- **Raise threshold** (e.g. 0.01 or 0.02) → fewer, stronger signals; often better P&L.
- **Lower commission** (e.g. 0.0005 or 0) to see if the signal has edge before adding cost.
- **Asymmetric thresholds:** set `BUY_THRESHOLD = 0.008` and `SELL_THRESHOLD = 0.015` so we sell only on stronger negative signals (reduces sell bias).
- Run the **threshold sweep** below to see which threshold gives the best P&L vs cash.

In [ ]:
# Threshold sweep: try several thresholds and report P&L (run after the backtest above)
if TENSORTRADE_AVAILABLE:
    results = []
    for thresh in [0.005, 0.01, 0.015, 0.02]:
        # Recreate lists (consumed in previous run)
        cl = close_arr.tolist()
        vl = np.expm1(log_vol_arr).tolist()
        v14 = vol_arr.tolist()
        pr = pred_ret_arr.tolist()
        price_s = Stream.source(cl, dtype="float").rename(f"USD-{ASSET.split('-')[0]}")
        ex = Exchange("ex", service=execute_order, options=ExchangeOptions(commission=COMMISSION))(price_s)
        c = Wallet(ex, 10000.0 * tt_instruments.USD)
        a = Wallet(ex, 0 * ASSET_INSTRUMENT)
        port = Portfolio(tt_instruments.USD, [c, a])
        feed = DataFeed([Stream.source(cl, dtype="float").rename("close"), Stream.source(vl, dtype="float").rename("vol"), Stream.source(v14, dtype="float").rename("v14"), Stream.source(pr, dtype="float").rename("pred_ret")])
        feed.compile()
        rwd = PBR(price=price_s)
        act = BSH(cash=c, asset=a).attach(rwd)
        env2 = default.create(feed=feed, portfolio=port, action_scheme=act, reward_scheme=rwd, window_size=5, max_allowed_loss=0.5)
        obs, _ = env2.reset()
        done, truncated, step = False, False, 0
        while not (done or truncated) and step < len(cl):
            action = 0 if pr[step] > thresh else (1 if pr[step] < -thresh else 2)
            obs, r, done, truncated, _ = env2.step(action)
            step += 1
        pnl_pct = 100 * (port.net_worth - 10000.0) / 10000.0
        results.append((thresh, pnl_pct, step))
    print("Threshold | P&L %  | Steps")
    for thresh, pnl_pct, steps in results:
        print(f"  {thresh:.3f}   | {pnl_pct:+.2f}% | {steps}")
    best = max(results, key=lambda x: x[1])
    print(f"Best in sweep: threshold {best[0]:.3f} → P&L {best[1]:+.2f}%")